## Data Ingestion Lab

In [22]:
!uv pip install chroma langchain langchain-openai langchain-text-splitters langchain-chroma langchain-community

Using Python 3.12.12 environment at: /Users/davidinyang-etoh/Projects/ai-projects/ai-playground/.venv
Audited 6 packages in 67ms


### Import modules and packages

In [23]:
import os
import glob
from pathlib import Path
from langchain_community.document_loaders import DirectoryLoader, TextLoader
from langchain_chroma import Chroma
from langchain_openai import OpenAIEmbeddings
from langchain_text_splitters import MarkdownTextSplitter

from dotenv import load_dotenv

load_dotenv(override=True)


True

In [24]:
MODEL = "gpt-4.1-nano"



EMBEDDING_MODEL = "text-embedding-3-small"

CHUNK_SIZE = 512
CHUNK_OVERLAP = 50

embeddings = OpenAIEmbeddings(model=EMBEDDING_MODEL)

In [25]:
DOMAIN = "topfaith.edu.ng"

DB_NAME = str(Path("db/vector_db"))
KNOWLEDGE_BASE_PATH = str(f"data/{DOMAIN}")

print(f"Loading documents from {KNOWLEDGE_BASE_PATH}")

Loading documents from data/topfaith.edu.ng


In [26]:
def fetch_documents():
    folders = glob.glob(str(Path(KNOWLEDGE_BASE_PATH)))
    documents = []
    for folder in folders:
        doc_type = os.path.basename(folder)
        loader = DirectoryLoader(
            folder, glob="**/*.md", loader_cls=TextLoader, loader_kwargs={"encoding": "utf-8"}
        )
        folder_docs = loader.load()
        for doc in folder_docs:
            doc.metadata["doc_type"] = doc_type
            documents.append(doc)
    return documents

documents = fetch_documents()

print(f"Found {len(documents)} documents in {KNOWLEDGE_BASE_PATH}")

Found 42 documents in data/topfaith.edu.ng


In [27]:
def create_chunks(documents):
    text_splitter = MarkdownTextSplitter(
        chunk_size=CHUNK_SIZE, chunk_overlap=CHUNK_OVERLAP)
    chunks = text_splitter.split_documents(documents)

    # Enrich each chunk with doc_type context prefix
    for chunk in chunks:
        doc_type = chunk.metadata.get("doc_type", "")
        source = chunk.metadata.get("source", "")
        filename = Path(source).stem if source else ""

        # Prepend context so the embedding captures it
        chunk.page_content = (
            f"[Document: {filename} | Category: {doc_type}]\n\n"
            + chunk.page_content
        )
    return chunks


In [28]:
def create_embeddings(chunks):
    if os.path.exists(DB_NAME):
        Chroma(persist_directory=DB_NAME,
               embedding_function=embeddings).delete_collection()

    vectorstore = Chroma.from_documents(
        documents=chunks, embedding=embeddings, persist_directory=DB_NAME
    )

    collection = vectorstore._collection
    count = collection.count()

    sample_embedding = collection.get(limit=1, include=["embeddings"])[
        "embeddings"][0]
    dimensions = len(sample_embedding)
    print(
        f"There are {count:,} vectors with {dimensions:,} dimensions in the vector store")
    return vectorstore

In [29]:
documents = fetch_documents()
chunks = create_chunks(documents)
create_embeddings(chunks)
print("Ingestion complete")

There are 448 vectors with 1,536 dimensions in the vector store
Ingestion complete


In [30]:
def get_chunks(query):
    vectorstore = Chroma(
        embedding_function=embeddings,
        persist_directory=DB_NAME
    )
    return vectorstore.similarity_search(query)


results = get_chunks("What is the purpose of the document?")

print(results)

[Document(id='4429ca56-ec9a-4e17-bc84-b78e8e661ebf', metadata={'doc_type': 'topfaith.edu.ng', 'source': 'data/topfaith.edu.ng/pg_about-us.md'}, page_content='[Document: pg_about-us | Category: topfaith.edu.ng]\n\n## Behind the vision'), Document(id='404c3da0-d877-44a4-9307-464a8ab65ff8', metadata={'source': 'data/topfaith.edu.ng/pg_about-us.md', 'doc_type': 'topfaith.edu.ng'}, page_content='[Document: pg_about-us | Category: topfaith.edu.ng]\n\n## Our Philosophy And Core Values'), Document(id='43ef1f5e-7ef2-47af-a8c1-c85cde6c52a9', metadata={'source': 'data/topfaith.edu.ng/pg_about-us.md', 'doc_type': 'topfaith.edu.ng'}, page_content='[Document: pg_about-us | Category: topfaith.edu.ng]\n\nUniversity incorporate and identify with all its stakeholders (staff, students, parents, and community) incorporating Faith, Academics, Community, Excellence, Service, Value, Integrity and symbolized with a strategic by-line as ‘FACES of Value and Integrity’ . This connotes the values associated with 